### TOAR-Classifier V2 – Inference Notebook
Run predictions on new stations using the saved model artifacts from training.

In [10]:
import sys
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [11]:
try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _SRC = Path().resolve()

_ROOT = _SRC.parent

for _p in [str(_ROOT), str(_SRC)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

In [12]:
from src.inference import TOARInference

#### 1. Load trained artifacts

In [15]:
MODELS_DIR = _ROOT / 'models'

# Use 'voting' for majority-vote ensemble, or specify a single model:
# 'rf', 'catboost', 'lgbm', 'xgb'
infer = TOARInference(models_dir=MODELS_DIR, best_model='voting', verbose=False)
infer

✅ Processor loaded from '/home/karim-mache/Documents/nouncode-ai/portfolio_projects/TOAR-Classifer-V2/models/processor.pkl'
✅ Feature engineer loaded from '/home/karim-mache/Documents/nouncode-ai/portfolio_projects/TOAR-Classifer-V2/models/feature_engineer.pkl'


TOARInference(models_dir='/home/karim-mache/Documents/nouncode-ai/portfolio_projects/TOAR-Classifer-V2/models', models=['rf', 'catboost', 'lgbm', 'xgb'], best_model='voting')

#### 2. Predict from station codes
Pass a single station code or a list of station codes. The data is fetched automatically from the TOAR-II API.

In [16]:
# Single station code
result_single = infer.predict('GB0682A')
result_single

,lat,lon,area_code,type_of_area,pred_voting
0,51.52253,-0.154611,GB0682A,urban,urban


In [17]:
# Multiple station codes (invalid codes are reported and skipped)
station_codes = ['GB0682A', 'FR04118', 'DENI051', 'INVALID_CODE_123', 'GB0013R', 'ES1422A']
result_codes = infer.predict(station_codes)
result_codes

⚠️  The following station code(s) could not be retrieved and were skipped — please verify them:
  ['INVALID_CODE_123', 'ES1422A']


,lat,lon,area_code,type_of_area,pred_voting
0,51.522530,-0.154611,GB0682A,urban,urban
1,48.870285,2.332503,FR04118,urban,urban
2,51.758160,10.612480,DENI051,rural,rural
3,50.597600,-3.716510,GB0013R,rural,rural
4,40.419167,-3.703333,28079035,urban,urban


#### 3. Predict from a raw DataFrame
Pass a DataFrame with raw station metadata (same format as the training CSV).

In [18]:
DATA_PATH = _ROOT / 'data' / 'stationglobalmetadata_12.09.2025.csv'
df_raw = pd.read_csv(DATA_PATH)
df_raw.shape

(24348, 30)

In [19]:
result_df = infer.predict(df_raw)
result_df.head(10)

,lat,lon,area_code,type_of_area,pred_voting
0,63.723200,-148.967600,02-068-0003,unknown,rural
1,46.547500,7.985000,CH0001G,unknown,rural
2,36.280000,100.900000,WLG,rural,rural
3,-40.683119,144.689939,CGO540S00,rural,rural
4,67.883333,21.066667,SE0013R,rural,rural
5,-14.247000,-170.564000,SMO514S00,rural,rural
6,35.692200,139.768900,jp13101010,urban,urban
7,-54.848400,-68.310700,AR0002G,rural,rural
8,-26.230594,28.020442,RSA024,urban,urban
9,-25.722592,28.420414,RSA005,suburban,suburban


#### 4. Predict from a JSON file
Pass a path to a `.json` file containing one station record (dict) or multiple records (list of dicts).

In [20]:
# Example: predict from a JSON file
# json_path = _ROOT / 'data' / 'new_stations.json'
# result_json = infer.predict(str(json_path))
# result_json

#### 5. Predict with all models (no best_model filter)
Create a new inference object without `best_model` to see predictions from every loaded classifier plus the voting ensemble.

In [22]:
infer_all = TOARInference(models_dir=MODELS_DIR, best_model=None, verbose=False)
result_all = infer_all.predict(station_codes)
result_all

✅ Processor loaded from '/home/karim-mache/Documents/nouncode-ai/portfolio_projects/TOAR-Classifer-V2/models/processor.pkl'
✅ Feature engineer loaded from '/home/karim-mache/Documents/nouncode-ai/portfolio_projects/TOAR-Classifer-V2/models/feature_engineer.pkl'
⚠️  The following station code(s) could not be retrieved and were skipped — please verify them:
  ['INVALID_CODE_123', 'ES1422A']


,lat,lon,area_code,type_of_area,pred_rf,pred_catboost,pred_lgbm,pred_xgb,pred_voting
0,51.522530,-0.154611,GB0682A,urban,urban,urban,urban,urban,urban
1,48.870285,2.332503,FR04118,urban,urban,urban,urban,urban,urban
2,51.758160,10.612480,DENI051,rural,rural,rural,rural,rural,rural
3,50.597600,-3.716510,GB0013R,rural,rural,rural,rural,rural,rural
4,40.419167,-3.703333,28079035,urban,urban,urban,urban,urban,urban


#### 6. Export results

In [23]:
# Uncomment to save predictions to CSV
# result_df.to_csv(_ROOT / 'data' / 'inference_predictions.csv', index=False)
# print('Saved predictions.')

#### 7. Predict new stations
Enter your own station codes below to classify them. Edit the list and run the cell.

In [24]:
# ✏️ Edit the list below with your station codes, then run this cell
my_station_codes = ['GB0682A', 'FR04118']

my_result = infer.predict(my_station_codes)
my_result

,lat,lon,area_code,type_of_area,pred_voting
0,51.522530,-0.154611,GB0682A,urban,urban
1,48.870285,2.332503,FR04118,urban,urban


### ============== END ==============